## PyINE traces dataset visualization demo

This notebook parses and displays samples/stats for a dataset of execution traces generated by our proposed framework.

In [ ]:
import collections
import importlib

import matplotlib.pyplot as plt

import pyine.data.traces.dataset_reader
import pyine.data.traces.dataset_utils
import pyine.utils.code.blocks
import pyine.utils.code.execution
import pyine.utils.portability

importlib.reload(pyine.data.traces.dataset_reader)
importlib.reload(pyine.data.traces.dataset_utils)
importlib.reload(pyine.utils.code.blocks)
importlib.reload(pyine.utils.code.execution)
importlib.reload(pyine.utils.portability)

In [ ]:
target_source_dataset = "TACO"  # name of the dataset whose traces we should visualize below
reader = pyine.data.traces.dataset_reader.DatasetReader(
    lmdb_path=pyine.data.traces.dataset_utils.get_latest_dataset_path(target_source_dataset),
)
print(f"{target_source_dataset} traces dataset contains {len(reader)} traces")

In [ ]:
# the lmdb dataset stores a ton of metadata related to when/how it was created; let's display some of it
metadata = reader.get_metadata()
metadata_fields_too_big_to_print = ["key_map", "installed_packages"]
print("traces dataset metadata:")
for metadata_key, metadata_value in metadata.items():
    if metadata_key in metadata_fields_too_big_to_print:  # too big to print, skip it
        continue
    print(f"\t{metadata_key}: {metadata_value}")

In [ ]:
# below, we will iterate over all traced solutions, gather some stats, and print them
traced_solution_tags = collections.Counter()
traced_function_count = 0
traced_block_count = 0
traced_line_count = 0

# for each traced solution in the dataset
for trace_idx, trace_data in enumerate(reader):
    # get the parent problem data for this specific solution
    problem_data = reader.get_problem_data(trace_idx)
    # extract traced steps that are 'in-context', i.e. inside the solution code string
    traced_steps = [t for t in trace_data.traced_steps if t is not None]
    traced_source_lines: set[int] = set()
    for t in traced_steps:
        if t.trace_key.file == pyine.utils.code.execution.EXEC_TRACE_FILE_NAME:
            traced_source_lines.add(t.trace_key.line)
    traced_line_count += len(traced_source_lines)
    for code_block_start_line, code_block_data in trace_data.code_blocks.items():
        if code_block_start_line not in traced_source_lines:
            continue
        traced_block_count += 1
        if code_block_data.type == pyine.utils.code.blocks.BlockType.FUNCTION:
            traced_function_count += 1
    # gather and count tags for the current traced solution
    traced_solution_tags.update(problem_data.problem_tags)

print(f"{traced_function_count=}")
print(f"{traced_block_count=}")
print(f"{traced_line_count=}")

traced_solution_tags = dict(sorted(traced_solution_tags.items(), key=lambda x: x[1], reverse=True))
plt.figure(figsize=(12, 6))
tags = list(traced_solution_tags.keys())
counts = list(traced_solution_tags.values())
plt.bar(tags, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Tags")
plt.ylabel("Count")
plt.title("Distribution of found tags")
plt.tight_layout()
plt.show()